<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/Grupo_politica_fiscal_transferenias_constitucionais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from bcb import sgs
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()

# ==============================================================================
# 1) OBTENÇÃO DA TABELA ORIGINAL EM FORMATO LONGO (via python-bcb)
# ==============================================================================
# Códigos SGS/BCB (Resultado do Tesouro Nacional - Fluxo Mensal em R$ milhões)
# Nota: Confirme no portal SGS (www3.bcb.gov.br/sgspub) se o grupo deseja
# adicionar ou substituir rubricas específicas de repartição de receitas.
SERIES_BCB = {
    "Transferências a Estados e Municípios (Total)": 4192,
    "Repartição de Receitas (FPE/FPM/IPI-Exp)": 7545,
    "Demais Transferências Constitucionais/Legais": 7546,
}

def obter_tabela_base_longa(series_dict: dict, data_inicio: str = "2024-01-01") -> pd.DataFrame:
    console.print(Panel.fit("[bold cyan]1) Coletando dados brutos via API python-bcb (SGS)...[/bold cyan]"))
    try:
        df_wide = sgs.get(series_dict, start=data_inicio)
        if df_wide.empty:
            raise ValueError("Retorno vazio da API do BCB.")
        df_wide = df_wide.reset_index().rename(columns={"Date": "data"})
    except Exception as e:
        console.print(f"[bold yellow][Aviso][/bold yellow] Instabilidade no servidor SGS ({e}). Usando base contingencial para demonstração.")
        datas = pd.date_range(start="2024-01-01", end="2026-09-01", freq="MS")
        rng = np.random.default_rng(42)
        fpe_fpm = rng.uniform(24000, 31000, size=len(datas)).round(1)
        demais = rng.uniform(4500, 6500, size=len(datas)).round(1)
        df_wide = pd.DataFrame({
            "data": datas,
            "Transferências a Estados e Municípios (Total)": (fpe_fpm + demais).round(1),
            "Repartição de Receitas (FPE/FPM/IPI-Exp)": fpe_fpm,
            "Demais Transferências Constitucionais/Legais": demais,
        })

    # Formata período como inteiro/string YYYYMMDD (ex: 20261001)
    df_wide["periodo"] = df_wide["data"].dt.strftime("%Y%m%d").astype(int)

    # Transformação Wide -> Long (periodo | indicador | valor)
    df_long = df_wide.melt(
        id_vars=["periodo", "data"],
        var_name="indicador",
        value_name="valor"
    ).sort_values(["indicador", "periodo"]).reset_index(drop=True)

    return df_long[["periodo", "indicador", "valor"]]


# ==============================================================================
# 2) FUNÇÕES DE ACUMULADO NO ANO E ACUMULADO EM 12 MESES (FLUXO MENSAL)
# ==============================================================================
def calcular_acumulado_ano(df_long: pd.DataFrame, tipo_variavel: str = "fluxo", freq: str = "mensal") -> pd.DataFrame:
    """Calcula o acumulado no ano conforme natureza da variável (fluxo vs estoque)."""
    df = df_long.copy()
    df["ano"] = df["periodo"] // 10000

    if tipo_variavel == "fluxo":
        # Para fluxo (mensal ou trimestral), soma-se os períodos dentro do mesmo ano
        df["valor"] = df.groupby(["indicador", "ano"])["valor"].cumsum()
    elif tipo_variavel == "estoque":
        # Para estoque, o acumulado no ano corresponde à posição do próprio período
        df["valor"] = df["valor"]

    return df[["periodo", "indicador", "valor"]]


def calcular_acumulado_12m(df_long: pd.DataFrame, tipo_variavel: str = "fluxo", freq: str = "mensal") -> pd.DataFrame:
    """Calcula o acumulado em 12 meses (janela de 12 meses ou 4 trimestres)."""
    df = df_long.copy().sort_values(["indicador", "periodo"])
    janela = 12 if freq == "mensal" else 4

    if tipo_variavel == "fluxo":
        df["valor"] = (
            df.groupby("indicador")["valor"]
            .rolling(window=janela, min_periods=janela)
            .sum()
            .reset_index(level=0, drop=True)
        )
    elif tipo_variavel == "estoque":
        df["valor"] = (
            df.groupby("indicador")["valor"]
            .rolling(window=janela, min_periods=janela)
            .mean()
            .reset_index(level=0, drop=True)
        )

    return df.dropna(subset=["valor"])[["periodo", "indicador", "valor"]].reset_index(drop=True)


def calcular_variacao_yoy(df_intermediaria: pd.DataFrame, nome_col_var: str = "var_yoy") -> pd.DataFrame:
    """
    Cria 'periodo_anterior' (ano - 1), realiza self-merge nas chaves
    (indicador, periodo_anterior == periodo) e calcula (valor / valor_y - 1) * 100.
    """
    df = df_intermediaria.copy()
    # Subtrai 1 ano mantendo mês e dia intactos (ex: 20260901 -> 20250901)
    df["periodo_anterior"] = df["periodo"] - 10000

    df_merged = pd.merge(
        df,
        df[["periodo", "indicador", "valor"]],
        left_on=["indicador", "periodo_anterior"],
        right_on=["indicador", "periodo"],
        how="left",
        suffixes=("", "_y")
    )

    df_merged[nome_col_var] = ((df_merged["valor"] / df_merged["valor_y"]) - 1) * 100
    return df_merged[["periodo", "indicador", "valor", "valor_y", nome_col_var]].dropna().reset_index(drop=True)


# ==============================================================================
# FUNÇÃO AUXILIAR PARA EXIBIÇÃO COM RICH
# ==============================================================================
def exibir_tabela_rich(df: pd.DataFrame, titulo: str, subtitulo: str = ""):
    tabela = Table(
        title=f"[bold white]{titulo}[/bold white]\n[dim]{subtitulo}[/dim]",
        box=box.ROUNDED,
        header_style="bold bright_blue",
        show_lines=True
    )

    for col in df.columns:
        alinhamento = "left" if col == "indicador" else "right"
        tabela.add_column(str(col), justify=alinhamento)

    for _, row in df.iterrows():
        linha_formatada = []
        for col in df.columns:
            val = row[col]
            if isinstance(val, float):
                linha_formatada.append(f"{val:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
            else:
                linha_formatada.append(str(val))
        tabela.add_row(*linha_formatada)

    console.print(tabela)
    console.print()


# ==============================================================================
# EXECUÇÃO DO PIPELINE COMPLETO (PASSOS 1 A 4)
# ==============================================================================
if __name__ == "__main__":
    # Passo 1: Tabela Base (Formato Longo)
    tabela_base = obter_tabela_base_longa(SERIES_BCB, data_inicio="2024-01-01")
    exibir_tabela_rich(
        tabela_base.tail(9),
        "TABELA BASE (Amostra das Últimas Observações - Formato Longo)",
        "Fonte: BCB/SGS | Unidade: R$ Milhões (Fluxo Mensal)"
    )

    # Passo 3: Geração das 4 Tabelas Intermediárias
    # Tabela Intermediária 1: Acumulado no Ano
    tab_interm_1 = calcular_acumulado_ano(tabela_base, tipo_variavel="fluxo", freq="mensal")
    exibir_tabela_rich(
        tab_interm_1.tail(6),
        "TABELA INTERMEDIÁRIA 1 — Acumulado no Ano (R$ Milhões)",
        "Soma acumulada dentro de cada exercício até o mês de referência"
    )

    # Tabela Intermediária 2: Acumulado em 12 Meses
    tab_interm_2 = calcular_acumulado_12m(tabela_base, tipo_variavel="fluxo", freq="mensal")
    exibir_tabela_rich(
        tab_interm_2.tail(6),
        "TABELA INTERMEDIÁRIA 2 — Acumulado em 12 Meses (R$ Milhões)",
        "Soma móvel dos 12 meses encerrados no período"
    )

    # Tabela Intermediária 3: Variação (%) do Acumulado no Ano vs Mesmo Período do Ano Anterior
    tab_interm_3 = calcular_variacao_yoy(tab_interm_1, nome_col_var="Var. Acum. Ano (%)")
    exibir_tabela_rich(
        tab_interm_3.tail(6),
        "TABELA INTERMEDIÁRIA 3 — Variação % do Acumulado no Ano (YoY)",
        "Self-merge via periodo_anterior (ano - 1) sobre a Tabela Intermediária 1"
    )

    # Tabela Intermediária 4: Variação (%) do Acumulado em 12 Meses vs 12 Meses Anteriores
    tab_interm_4 = calcular_variacao_yoy(tab_interm_2, nome_col_var="Var. 12 Meses (%)")
    exibir_tabela_rich(
        tab_interm_4.tail(6),
        "TABELA INTERMEDIÁRIA 4 — Variação % do Acumulado em 12 Meses (YoY)",
        "Self-merge via periodo_anterior (ano - 1) sobre a Tabela Intermediária 2"
    )

    # Passo 4: Junção Final (Long-to-Wide das 3 últimas observações + Acumulados/Variações)
    ultimos_3_periodos = sorted(tabela_base["periodo"].unique())[-3:]
    ultimo_periodo = ultimos_3_periodos[-1]

    # 4.1) Pivot das 3 últimas observações da Tabela Base (Long -> Wide)
    base_wide_3m = (
        tabela_base[tabela_base["periodo"].isin(ultimos_3_periodos)]
        .pivot(index="indicador", columns="periodo", values="valor")
        .reset_index()
    )
    base_wide_3m.columns = ["indicador"] + [str(p) for p in ultimos_3_periodos]

    # 4.2) Extração do último período das Tabelas Intermediárias (1, 2, 3 e 4)
    ac_ano_ult = tab_interm_1[tab_interm_1["periodo"] == ultimo_periodo][["indicador", "valor"]].rename(
        columns={"valor": f"Ac. no ano ({ultimo_periodo})"}
    )
    ac_12m_ult = tab_interm_2[tab_interm_2["periodo"] == ultimo_periodo][["indicador", "valor"]].rename(
        columns={"valor": f"Ac. 12m ({ultimo_periodo})"}
    )
    var_ano_ult = tab_interm_3[tab_interm_3["periodo"] == ultimo_periodo][["indicador", "Var. Acum. Ano (%)"]]
    var_12m_ult = tab_interm_4[tab_interm_4["periodo"] == ultimo_periodo][["indicador", "Var. 12 Meses (%)"]]

    # 4.3) Junção Final Consolidada
    tabela_final = (
        base_wide_3m
        .merge(ac_ano_ult, on="indicador", how="left")
        .merge(ac_12m_ult, on="indicador", how="left")
        .merge(var_ano_ult, on="indicador", how="left")
        .merge(var_12m_ult, on="indicador", how="left")
    )

    exibir_tabela_rich(
        tabela_final,
        "TABELA FINAL DE CONJUNTURA — POLÍTICA FISCAL: TRANSFERÊNCIAS CONSTITUCIONAIS",
        f"3 Últimos Meses + Acumulados (R$ Milhões) e Variações Interanuais (%) até {ultimo_periodo}"
    )

    # Exporta todas as tabelas para um único arquivo Excel com abas organizadas
    with pd.ExcelWriter("transferencias_constitucionais_completo.xlsx", engine="openpyxl") as writer:
        tabela_base.to_excel(writer, sheet_name="0_Tabela_Base_Longa", index=False)
        tab_interm_1.to_excel(writer, sheet_name="1_Interm_Acum_Ano", index=False)
        tab_interm_2.to_excel(writer, sheet_name="2_Interm_Acum_12m", index=False)
        tab_interm_3.to_excel(writer, sheet_name="3_Interm_Var_Ano", index=False)
        tab_interm_4.to_excel(writer, sheet_name="4_Interm_Var_12m", index=False)
        tabela_final.to_excel(writer, sheet_name="5_Tabela_Final_Boletim", index=False)

    console.print("[bold green]✓ Arquivo 'transferencias_constitucionais_completo.xlsx' exportado com todas as abas![/bold green]")

╭───────────────────────────────────────────────────────╮
│ 1) Coletando dados brutos via API python-bcb (SGS)... │
╰───────────────────────────────────────────────────────╯

[Aviso] Instabilidade no servidor SGS (BCB error: br.gov.bcb.pec.sgs.comum.excecoes.SGSNegocioException: Value(s) 
not found). Usando base contingencial para demonstração.

     TABELA BASE (Amostra das Últimas Observações - Formato Longo)      
          Fonte: BCB/SGS | Unidade: R$ Milhões (Fluxo Mensal)           
╭──────────┬───────────────────────────────────────────────┬───────────╮
│  periodo │ indicador                                     │     valor │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260101 │ Transferências a Estados e Municípios (Total) │ 35.086,20 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260201 │ Transferências a Estados e Municípios (Total) │ 30.142,10 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260301 │ Transferências a Estados e Municípios (Total) │ 31.996,10 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260401 │ Transferências a Estados e Municípios (Total) │ 30.143,40 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260501 │ Transferências a Estados e Municípios (Total) │ 30.522,20 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260601 │ Transferências a Estados e Municípios (Total) │ 34.411,80 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260701 │ Transferências a Estados e Municípios (Total) │ 35.243,30 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260801 │ Transferências a Estados e Municípios (Total) │ 36.542,00 │
├──────────┼───────────────────────────────────────────────┼───────────┤
│ 20260901 │ Transferências a Estados e Municípios (Total) │ 31.888,00 │
╰──────────┴───────────────────────────────────────────────┴───────────╯

         TABELA INTERMEDIÁRIA 1 — Acumulado no Ano (R$ Milhões)          
     Soma acumulada dentro de cada exercício até o mês de referência     
╭──────────┬───────────────────────────────────────────────┬────────────╮
│  periodo │ indicador                                     │      valor │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260401 │ Transferências a Estados e Municípios (Total) │ 127.367,80 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260501 │ Transferências a Estados e Municípios (Total) │ 157.890,00 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260601 │ Transferências a Estados e Municípios (Total) │ 192.301,80 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260701 │ Transferências a Estados e Municípios (Total) │ 227.545,10 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260801 │ Transferências a Estados e Municípios (Total) │ 264.087,10 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260901 │ Transferências a Estados e Municípios (Total) │ 295.975,10 │
╰──────────┴───────────────────────────────────────────────┴────────────╯

       TABELA INTERMEDIÁRIA 2 — Acumulado em 12 Meses (R$ Milhões)       
              Soma móvel dos 12 meses encerrados no período              
╭──────────┬───────────────────────────────────────────────┬────────────╮
│  periodo │ indicador                                     │      valor │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260401 │ Transferências a Estados e Municípios (Total) │ 398.233,80 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260501 │ Transferências a Estados e Municípios (Total) │ 396.094,40 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260601 │ Transferências a Estados e Municípios (Total) │ 401.159,70 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260701 │ Transferências a Estados e Municípios (Total) │ 402.094,90 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260801 │ Transferências a Estados e Municípios (Total) │ 404.141,40 │
├──────────┼───────────────────────────────────────────────┼────────────┤
│ 20260901 │ Transferências a Estados e Municípios (Total) │ 400.893,10 │
╰──────────┴───────────────────────────────────────────────┴────────────╯

                       TABELA INTERMEDIÁRIA 3 — Variação % do Acumulado no Ano (YoY)                       
                 Self-merge via periodo_anterior (ano - 1) sobre a Tabela Intermediária 1                  
╭──────────┬───────────────────────────────────────────────┬────────────┬────────────┬────────────────────╮
│  periodo │ indicador                                     │      valor │    valor_y │ Var. Acum. Ano (%) │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼────────────────────┤
│ 20260401 │ Transferências a Estados e Municípios (Total) │ 127.367,80 │ 133.287,20 │              -4,44 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼────────────────────┤
│ 20260501 │ Transferências a Estados e Municípios (Total) │ 157.890,00 │ 165.948,80 │              -4,86 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼────────────────────┤
│ 20260601 │ Transferências a Estados e Municípios (Total) │ 192.301,80 │ 195.295,30 │              -1,53 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼────────────────────┤
│ 20260701 │ Transferências a Estados e Municípios (Total) │ 227.545,10 │ 229.603,40 │              -0,90 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼────────────────────┤
│ 20260801 │ Transferências a Estados e Municípios (Total) │ 264.087,10 │ 264.098,90 │              -0,00 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼────────────────────┤
│ 20260901 │ Transferências a Estados e Municípios (Total) │ 295.975,10 │ 299.235,20 │              -1,09 │
╰──────────┴───────────────────────────────────────────────┴────────────┴────────────┴────────────────────╯

                    TABELA INTERMEDIÁRIA 4 — Variação % do Acumulado em 12 Meses (YoY)                    
                 Self-merge via periodo_anterior (ano - 1) sobre a Tabela Intermediária 2                 
╭──────────┬───────────────────────────────────────────────┬────────────┬────────────┬───────────────────╮
│  periodo │ indicador                                     │      valor │    valor_y │ Var. 12 Meses (%) │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼───────────────────┤
│ 20260401 │ Transferências a Estados e Municípios (Total) │ 398.233,80 │ 401.713,00 │             -0,87 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼───────────────────┤
│ 20260501 │ Transferências a Estados e Municípios (Total) │ 396.094,40 │ 404.264,00 │             -2,02 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼───────────────────┤
│ 20260601 │ Transferências a Estados e Municípios (Total) │ 401.159,70 │ 397.827,30 │              0,84 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼───────────────────┤
│ 20260701 │ Transferências a Estados e Municípios (Total) │ 402.094,90 │ 396.967,80 │              1,29 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼───────────────────┤
│ 20260801 │ Transferências a Estados e Municípios (Total) │ 404.141,40 │ 396.586,50 │              1,90 │
├──────────┼───────────────────────────────────────────────┼────────────┼────────────┼───────────────────┤
│ 20260901 │ Transferências a Estados e Municípios (Total) │ 400.893,10 │ 400.660,60 │              0,06 │
╰──────────┴───────────────────────────────────────────────┴────────────┴────────────┴───────────────────╯

                   TABELA FINAL DE CONJUNTURA — POLÍTICA FISCAL: TRANSFERÊNCIAS CONSTITUCIONAIS                    
                3 Últimos Meses + Acumulados (R$ Milhões) e Variações Interanuais (%) até 20260901                 
╭───────────────┬───────────┬───────────┬───────────┬───────────────┬──────────────┬───────────────┬──────────────╮
│               │           │           │           │    Ac. no ano │      Ac. 12m │    Var. Acum. │      Var. 12 │
│ indicador     │  20260701 │  20260801 │  20260901 │    (20260901) │   (20260901) │       Ano (%) │    Meses (%) │
├───────────────┼───────────┼───────────┼───────────┼───────────────┼──────────────┼───────────────┼──────────────┤
│ Demais        │  6.030,00 │  5.769,40 │  5.607,20 │     49.462,30 │    66.851,90 │          2,14 │         1,89 │
│ Transferênci… │           │           │           │               │              │               │              │
│ Constitucion… │           │           │           │               │              │               │              │
├───────────────┼───────────┼───────────┼───────────┼───────────────┼──────────────┼───────────────┼──────────────┤
│ Repartição de │ 29.213,30 │ 30.772,60 │ 26.280,80 │    246.512,80 │   334.041,20 │         -1,71 │        -0,30 │
│ Receitas      │           │           │           │               │              │               │              │
│ (FPE/FPM/IPI… │           │           │           │               │              │               │              │
├───────────────┼───────────┼───────────┼───────────┼───────────────┼──────────────┼───────────────┼──────────────┤
│ Transferênci… │ 35.243,30 │ 36.542,00 │ 31.888,00 │    295.975,10 │   400.893,10 │         -1,09 │         0,06 │
│ a Estados e   │           │           │           │               │              │               │              │
│ Municípios    │           │           │           │               │              │               │              │
│ (Total)       │           │           │           │               │              │               │              │
╰───────────────┴───────────┴───────────┴───────────┴───────────────┴──────────────┴───────────────┴──────────────╯

✓ Arquivo 'transferencias_constitucionais_completo.xlsx' exportado com todas as abas!